# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [ ]:
import pandas as pd
import numpy as np

u1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
u2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
u3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

def clean_col(name: str) -> str:
    name = name.strip().lower()
    name = pd.Series(name).str.normalize("NFKD").str.encode("ascii","ignore").str.decode("ascii").iloc[0]
    name = pd.Series(name).str.replace(r"[^0-9a-zA-Z]+", "_", regex=True).iloc[0]
    return name.strip("_")

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_col(c) for c in df.columns]
    return df

df1 = clean_columns(pd.read_csv(u1))
df2 = clean_columns(pd.read_csv(u2))
df3 = clean_columns(pd.read_csv(u3))

all_cols = sorted(set(df1.columns) | set(df2.columns) | set(df3.columns))
df1 = df1.reindex(columns=all_cols)
df2 = df2.reindex(columns=all_cols)
df3 = df3.reindex(columns=all_cols)

combined = pd.concat([df1, df2, df3], ignore_index=True)

print("Shapes:", df1.shape, df2.shape, df3.shape)
print("Combined shape:", combined.shape)
combined.head(3)


Shapes: (4008, 12) (996, 12) (7070, 12)
Combined shape: (12074, 12)


,customer,customer_lifetime_value,education,gender,income,monthly_premium_auto,number_of_open_complaints,policy_type,st,state,total_claim_amount,vehicle_class
0,RB50392,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Washington,NaN,2.704934,Four-Door Car
1,QZ44356,697953.59%,Bachelor,F,0.0,94.0,1/0/00,Personal Auto,Arizona,NaN,1131.464935,Four-Door Car
2,AI49188,1288743.17%,Bachelor,F,48767.0,108.0,1/0/00,Personal Auto,Nevada,NaN,566.472247,Two-Door Car


In [ ]:

obj_cols = combined.select_dtypes(include="object").columns
for c in obj_cols:
    combined[c] = combined[c].astype(str).str.strip()

if "state" in combined.columns:
    combined["state"] = combined["state"].str.title()
if "gender" in combined.columns:
    combined["gender"] = (combined["gender"].str.lower()
                          .replace({"m":"male","male":"male","f":"female","female":"female"}))
if "policy_type" in combined.columns:
    combined["policy_type"] = combined["policy_type"].str.title()
if "vehicle_class" in combined.columns:
    combined["vehicle_class"] = combined["vehicle_class"].str.title()

def to_numeric_clean(s: pd.Series) -> pd.Series:
    return (s.astype(str)
              .str.replace(r"[^0-9.\-]", "", regex=True)
              .replace({"": np.nan, "-": np.nan})
              .astype(float))

for col in ["customer_lifetime_value","income","monthly_premium_auto","total_claim_amount",
            "number_of_open_complaints","number_of_policies"]:
    if col in combined.columns:
        combined[col] = to_numeric_clean(combined[col])


for dcol in ["effective_to_date", "date"]:
    if dcol in combined.columns:
        combined[dcol] = pd.to_datetime(combined[dcol], errors="coerce")


before = combined.shape[0]
combined = combined.drop_duplicates()
after = combined.shape[0]
print(f"Removed {before-after} duplicate rows")

combined.info()


Removed 2939 duplicate rows
<class 'pandas.core.frame.DataFrame'>
Index: 9135 entries, 0 to 12073
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer                   9135 non-null   object 
 1   customer_lifetime_value    9127 non-null   float64
 2   education                  9135 non-null   object 
 3   gender                     9135 non-null   object 
 4   income                     9134 non-null   float64
 5   monthly_premium_auto       9134 non-null   float64
 6   number_of_open_complaints  9134 non-null   float64
 7   policy_type                9135 non-null   object 
 8   st                         9135 non-null   object 
 9   state                      9135 non-null   object 
 10  total_claim_amount         9134 non-null   float64
 11  vehicle_class              9135 non-null   object 
dtypes: float64(5), object(7)
memory usage: 927.8+ KB


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [3]:
m_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
mkt = pd.read_csv(m_url)
mkt.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in mkt.columns]

for col in ["customer_lifetime_value","income","monthly_premium_auto","total_claim_amount"]:
    if col in mkt.columns:
        mkt[col] = pd.to_numeric(mkt[col], errors="coerce")
if "effective_to_date" in mkt.columns:
    mkt["effective_to_date"] = pd.to_datetime(mkt["effective_to_date"], errors="coerce")

mkt.head(3)


/var/folders/cv/pfmqj_4n6fg_b8vt_pytwxwr0000gn/T/ipykernel_25404/3769050212.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  mkt["effective_to_date"] = pd.to_datetime(mkt["effective_to_date"], errors="coerce")


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [ ]:

if {"state","policy_type","monthly_premium_auto"} <= set(mkt.columns):
    pivot_premium = pd.pivot_table(
        mkt,
        values="monthly_premium_auto",
        index="state",
        columns="policy_type",
        aggfunc="sum",
        fill_value=0
    )
    print("Total monthly premium by State x Policy Type:")
    display(pivot_premium)

if {"state","policy_type","total_claim_amount"} <= set(mkt.columns):
    pivot_claims = pd.pivot_table(
        mkt,
        values="total_claim_amount",
        index="state",
        columns="policy_type",
        aggfunc="sum",
        fill_value=0
    )
    print("Total claim amount by State x Policy Type:")
    display(pivot_claims)


Total monthly premium by State x Policy Type:


policy_type,Corporate Auto,Personal Auto,Special Auto
state,,,
Arizona,34702,135236,7803
California,76737,243932,11304
Nevada,21435,69529,3360
Oregon,55870,203556,13472
Washington,16250,62683,2597


Total claim amount by State x Policy Type:


policy_type,Corporate Auto,Personal Auto,Special Auto
state,,,
Arizona,159846.763167,6.200631e+05,38128.764306
California,356758.850189,1.154621e+06,53053.832584
Nevada,98059.766371,3.270379e+05,15022.258539
Oregon,254020.034328,9.441617e+05,64374.181371
Washington,75153.958132,3.063842e+05,12248.793802


In [5]:
if {"education","customer_lifetime_value"} <= set(mkt.columns):
    pivot_clv_edu = pd.pivot_table(
        mkt,
        values="customer_lifetime_value",
        index="education",
        aggfunc="mean"
    ).sort_values(by="customer_lifetime_value", ascending=False)

    print("Average Customer Lifetime Value by Education:")
    display(pivot_clv_edu)


Average Customer Lifetime Value by Education:


,customer_lifetime_value
education,
High School or Below,8415.293805
Master,8162.524581
College,7900.072700
Bachelor,7792.273822
Doctor,7372.029537


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here